In [2]:
import os
import pandas as pd
import numpy as np
import scipy.signal as signal

resampled_dir = '../data/kaggle-drdataboston/resampled_subjects'
output_npz = '../data/kaggle-drdataboston/10step_windows_with_gyro.npz'
fs = 50
steps_per_window = 10
samples_per_step = int(fs * 0.6)

X = []
filenames = []

for filename in os.listdir(resampled_dir):
    if not filename.endswith('.csv'):
        continue

    df = pd.read_csv(os.path.join(resampled_dir, filename))
    if not all(col in df.columns for col in [
        'motionUserAccelerationX.G.', 'motionUserAccelerationY.G.', 'motionUserAccelerationZ.G.',
        'gyroRotationX.rad.s.', 'gyroRotationY.rad.s.', 'gyroRotationZ.rad.s.'
    ]):
        continue

    acc_mag = np.sqrt(
        df['motionUserAccelerationX.G.']**2 +
        df['motionUserAccelerationY.G.']**2 +
        df['motionUserAccelerationZ.G.']**2
    )
    b, a = signal.butter(2, [0.5 / (fs/2), 3.0 / (fs/2)], btype='band')
    acc_filt = signal.filtfilt(b, a, acc_mag)
    peaks, _ = signal.find_peaks(acc_filt, distance=fs*0.4)

    for i in range(len(peaks) - steps_per_window):
        start = peaks[i]
        end = peaks[i + steps_per_window]

        if end <= start or end > len(df):
            continue

        segment = df.iloc[start:end][[
            'motionUserAccelerationX.G.',
            'motionUserAccelerationY.G.',
            'motionUserAccelerationZ.G.',
            'gyroRotationX.rad.s.',
            'gyroRotationY.rad.s.',
            'gyroRotationZ.rad.s.'
        ]].to_numpy()

        if segment.shape[0] < 30:
            continue

        resampled = signal.resample(segment, 200)  # (200, 6)
        flat = resampled.flatten()  # (1200,)
        X.append(flat)
        filenames.append(filename)

    print(f"Processed {filename} → total windows so far: {len(X)}")

# Convert to arrays
X = np.array(X)
filenames = np.array(filenames)

# Save compressed
np.savez_compressed(output_npz, X=X, filenames=filenames)
print(f"✅ Saved {X.shape[0]} windows to {output_npz}")


Processed sub58-lw-s1.csv → total windows so far: 332
Processed sub92-lw-s2.csv → total windows so far: 637
Processed sub93-rp-s1.csv → total windows so far: 987
Processed sub56-lw-s2.csv → total windows so far: 1257
Processed sub43-rp-s2.csv → total windows so far: 1554
Processed sub32-lw-s2.csv → total windows so far: 1860
Processed sub20-rp-s2.csv → total windows so far: 2209
Processed sub38-lw-s1.csv → total windows so far: 2647
Processed sub26-rp-s1.csv → total windows so far: 2981
Processed sub79-lw-s2.csv → total windows so far: 3322
Processed sub21-rp-s2.csv → total windows so far: 3664
Processed sub33-rp-s1.csv → total windows so far: 3983
Processed sub37-lw-s1.csv → total windows so far: 4331
Processed sub20-rp-s1.csv → total windows so far: 4642
Processed sub40-lw-s1.csv → total windows so far: 4999
Processed sub24-lw-s1.csv → total windows so far: 5392
Processed sub62-lw-s2.csv → total windows so far: 5654
Processed sub31-lw-s2.csv → total windows so far: 5953
Processed sub

In [3]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from tensorflow.keras import layers, models, callbacks
import joblib
import os

# --- Load data ---
base_path = '../data/kaggle-drdataboston/'
data = np.load(os.path.join(base_path, '10step_windows_with_gyro.npz'))
X = data['X']  # shape: (n_windows, 1200)
filenames = data['filenames']  # shape: (n_windows,)

print("✅ Loaded data:", X.shape)

# --- Scale input ---
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# --- Define autoencoder ---
input_dim = X_scaled.shape[1]
latent_dim = 32  # more context than 5-step version

input_layer = layers.Input(shape=(input_dim,))
encoded = layers.Dense(256, activation='relu')(input_layer)
encoded = layers.Dense(128, activation='relu')(encoded)
latent = layers.Dense(latent_dim, activation='relu')(encoded)
decoded = layers.Dense(128, activation='relu')(latent)
decoded = layers.Dense(256, activation='relu')(decoded)
output_layer = layers.Dense(input_dim, activation='linear')(decoded)

autoencoder = models.Model(inputs=input_layer, outputs=output_layer)
autoencoder.compile(optimizer='adam', loss='mse')

# --- Train ---
es = callbacks.EarlyStopping(patience=5, restore_best_weights=True)
autoencoder.fit(X_scaled, X_scaled, epochs=50, batch_size=128, validation_split=0.1, callbacks=[es])

# --- Extract encoder ---
encoder = models.Model(inputs=input_layer, outputs=latent)
X_encoded = encoder.predict(X_scaled)
print("✅ Encoded shape:", X_encoded.shape)

# --- Save everything ---
encoder_path = os.path.join(base_path, "encoder_10steps_with_gyro.keras")
scaler_path = os.path.join(base_path, "scaler_10steps_with_gyro.pkl")
encoded_path = os.path.join(base_path, "X_encoded_10steps_with_gyro.npy")
filenames_path = os.path.join(base_path, "filenames_10steps_with_gyro.npy")

encoder.save(encoder_path)
joblib.dump(scaler, scaler_path)
np.save(encoded_path, X_encoded)
np.save(filenames_path, filenames)

print("✅ Saved:")
print(" - Encoder:", encoder_path)
print(" - Scaler:", scaler_path)
print(" - Encoded vectors:", encoded_path)
print(" - Filenames:", filenames_path)


2025-03-30 18:50:31.780235: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-30 18:50:31.791220: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-30 18:50:31.869022: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2025-03-30 18:50:31.931333: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743349831.987764   26075 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743349832.00

✅ Loaded data: (88292, 1200)


2025-03-30 18:50:38.651828: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Epoch 1/50
621/621 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.4091 - val_loss: 0.2517
Epoch 2/50
621/621 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.2364 - val_loss: 0.2272
Epoch 3/50
621/621 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.1935 - val_loss: 0.2289
Epoch 4/50
621/621 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.2034 - val_loss: 0.2134
Epoch 5/50
621/621 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.1723 - val_loss: 0.2287
Epoch 6/50
621/621 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.1812 - val_loss: 0.2046
Epoch 7/50
621/621 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - loss: 0.1732 - val_loss: 0.2078
Epoch 8/50
621/621 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - loss: 0.1589 - val_loss: 0.1989
Epoch 9/50
621/621 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - loss: 0.1533 - val_loss: 0.2151
Epoch 10/50
621/621 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - loss: 0.1676 - val_loss: 0.1990
Epoch 11/50
621/621 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms/step - loss: 0.1587 - val_loss: 0.1968
Epoch 12/50
621/621 ━━━━━━━━━━━━━━━━━━━━ 6s 10ms

Encoded shape: (342, 16)


/home/robert/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:690: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(
